# Tamil TTS POC - Kaggle

1. Clone repo from GitHub
2. Install dependencies
3. Generate Tamil speech with Jaya voice
4. Save output to /kaggle/working (auto-downloadable)

**Prerequisites:** Phone-verify your Kaggle account (Settings → Phone Verification) to enable internet access.

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/poc_colab"
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

result = subprocess.run(
    ["git", "clone", "--branch", "main", "--single-branch",
     "https://github.com/saravmani-kmu/poc_colab.git", REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("ERROR: git clone failed!")
    print(result.stderr)
    raise RuntimeError("Internet access required. Verify your phone at kaggle.com/settings")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

In [ ]:
# Install dependencies
!pip install transformers==4.46.1 soundfile sentencepiece "protobuf>=5.26.1,<6.0" -q
!pip install flatten_dict julius argbind randomname pyloudnorm pystoi markdown2 fire -q
!pip install descript-audio-codec descript-audiotools --no-deps -q
!pip install git+https://github.com/huggingface/parler-tts.git --no-deps -q
!pip install "protobuf>=5.26.1,<6.0" -q

In [ ]:
# Login to HuggingFace for gated model access
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    secret = UserSecretsClient()
    hf_token = secret.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle secret")
except Exception as e:
    print(f"Could not get Kaggle secret: {e}")
    print("Set HF_TOKEN in Kaggle Secrets (Add-ons > Secrets)")

In [ ]:
# Run the TTS pipeline
os.chdir("/kaggle/working/poc_colab")
!python main.py

In [ ]:
# Play generated audio
from IPython.display import Audio, display

output_dir = "/kaggle/working/poc_colab/output"
for f in sorted(os.listdir(output_dir)):
    if f.endswith(".wav"):
        print(f"\n--- {f} ---")
        display(Audio(os.path.join(output_dir, f), autoplay=False))

In [ ]:
# Copy outputs to /kaggle/working so they appear as downloadable output
import shutil

output_dest = "/kaggle/working/output"
if os.path.exists(output_dest):
    shutil.rmtree(output_dest)
shutil.copytree("/kaggle/working/poc_colab/output", output_dest)
shutil.copy2("/kaggle/working/poc_colab/output_bundle.zip", "/kaggle/working/output_bundle.zip")

print("Output files (downloadable from Kaggle):")
for f in os.listdir("/kaggle/working"):
    if f != "poc_colab":
        print(f"  {f}")